# IC Design SFT — 评估结果分析
基于 `assets/eval_result.json`，对比基座模型与微调模型在 IC Design 测试集上的表现。

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
plt.style.use('seaborn-v0_8-whitegrid')

with open('../assets/eval_result.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

base_scores = [item['rouge_l'] for item in data['base_results']]
ft_scores   = [item['rouge_l'] for item in data['ft_results']]

print(f"测试集样本数:      {len(base_scores)}")
print(f"基座模型 ROUGE-L:  {np.mean(base_scores):.4f}")
print(f"微调模型 ROUGE-L:  {np.mean(ft_scores):.4f}")
print(f"提升幅度:          +{np.mean(ft_scores) - np.mean(base_scores):.4f} "
      f"({(np.mean(ft_scores)/np.mean(base_scores)-1)*100:.1f}%)")

## 1. 整体对比：基座 vs 微调

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

labels = ['Base Model', 'Fine-tuned Model']
means  = [np.mean(base_scores), np.mean(ft_scores)]
colors = ['#9ecae1', '#2171b5']

bars = ax.bar(labels, means, color=colors, width=0.4, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, max(means) * 1.2)
ax.set_ylabel('Average ROUGE-L (F1)', fontsize=11)
ax.set_title('Base vs Fine-tuned: ROUGE-L on IC Design Test Set', fontsize=12)
plt.tight_layout()
plt.savefig('../assets/results.png', dpi=150, bbox_inches='tight')
plt.show()
print('已保存至 assets/results.png')

## 2. ROUGE-L 分布对比（直方图）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, scores, label, color in zip(
    axes,
    [base_scores, ft_scores],
    ['Base Model', 'Fine-tuned Model'],
    ['#9ecae1', '#2171b5']
):
    ax.hist(scores, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(scores), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {np.mean(scores):.4f}')
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('ROUGE-L Score', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.legend(fontsize=10)

fig.suptitle('ROUGE-L Score Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. 逐样本提升散点图

In [ ]:
improvements = np.array(ft_scores) - np.array(base_scores)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2171b5' if v >= 0 else '#cb181d' for v in improvements]
ax.bar(range(len(improvements)), improvements, color=colors, width=1.0, linewidth=0)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(np.mean(improvements), color='orange', linestyle='--',
           linewidth=1.5, label=f'Mean: +{np.mean(improvements):.4f}')

improved = (improvements > 0).sum()
ax.set_title(f'Per-sample ROUGE-L Improvement  '
             f'(improved: {improved}/{len(improvements)}, {improved/len(improvements)*100:.1f}%)',
             fontsize=12)
ax.set_xlabel('Sample Index', fontsize=10)
ax.set_ylabel('ROUGE-L Delta (FT - Base)', fontsize=10)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 4. 最受益 / 最受损的样本（Top 5）

In [ ]:
indexed = sorted(enumerate(improvements), key=lambda x: x[1])
worst5  = indexed[:5]
best5   = indexed[-5:][::-1]

print('=== 提升最大的 5 条 ===')
for rank, (i, delta) in enumerate(best5, 1):
    inst = data['ft_results'][i]['instruction']
    print(f"{rank}. [{delta:+.4f}] {inst[:60]}...")

print('\n=== 下降最多的 5 条 ===')
for rank, (i, delta) in enumerate(worst5, 1):
    inst = data['ft_results'][i]['instruction']
    print(f"{rank}. [{delta:+.4f}] {inst[:60]}...")

## 5. 查看单条样本详情

In [ ]:
# 修改 idx 查看任意一条样本
idx = best5[0][0]

base_item = data['base_results'][idx]
ft_item   = data['ft_results'][idx]

print(f"【指令】\n{ft_item['instruction']}")
print(f"\n【参考答案】\n{ft_item['reference'][:300]}...")
print(f"\n【基座模型输出】(ROUGE-L={base_item['rouge_l']:.4f})\n{base_item['prediction'][:300]}...")
print(f"\n【微调模型输出】(ROUGE-L={ft_item['rouge_l']:.4f})\n{ft_item['prediction'][:300]}...")